In [1]:
from src.metric import TimestampMetric
from src.helpers import normalize_project, get_project_folders, get_timestamps
import polars as pl

In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/code-results-9-4-2026"
OUTPUT_FOLDER = "../results"

In [3]:
metric_truck_factor = TimestampMetric("doa_truck_factor")
metric_decayed_truck_factor = TimestampMetric("doa_decayed_truck_factor")
metric_gini_coefficient = TimestampMetric("doa_gini_coefficient")
metric_decayed_gini_coefficient = TimestampMetric("doa_decayed_gini_coefficient")
metric_author_count = TimestampMetric("author_count")
metric_doa_author_count = TimestampMetric("doa_author_count")

for project in get_project_folders(INPUT_FOLDER):
    project_name = normalize_project(project.name)
    df = pl.read_csv(
        f"{project}/truck_stats.csv",
        infer_schema_length=10000,
        schema_overrides={
            "gini_coefficient": pl.Float64,
            "gini_coefficient_decay": pl.Float64,
        },
    )
    timestamps = get_timestamps(project)

    for item in df.iter_rows(named=True):
        date = timestamps[item["index"]]
        metric_truck_factor.add(project_name, date, item["truck_factor"])
        metric_decayed_truck_factor.add(project_name, date, item["truck_factor_decay"])
        metric_gini_coefficient.add(project_name, date, item["gini_coefficient"])
        metric_decayed_gini_coefficient.add(project_name, date, item["gini_coefficient_decay"])
        metric_author_count.add(project_name, date, item["num_authors"])
        metric_doa_author_count.add(project_name, date, item["num_doa_authors"])

metric_truck_factor.save(OUTPUT_FOLDER)
metric_decayed_truck_factor.save(OUTPUT_FOLDER)
metric_gini_coefficient.save(OUTPUT_FOLDER)
metric_decayed_gini_coefficient.save(OUTPUT_FOLDER)
metric_author_count.save(OUTPUT_FOLDER)
metric_doa_author_count.save(OUTPUT_FOLDER)